## В выводе каждой ячейки стоит номер задачи

In [2]:
from fractions import Fraction as F
import numpy as np

In [3]:
L = np.linalg; R = np.random.default_rng(0); eq = np.allclose; d, iv = L.det, L.inv

I = lambda a: np.rint(a).astype(int).tolist()      
D = lambda a: (np.round(a, 4) + 0).tolist()       
Q = lambda M: [[int(x) for x in r] for r in M]      

## Функции

In [4]:
def rref(a):
    a = [[F(x) for x in r] for r in a]; n, m = len(a), len(a[0]); p = []
    for c in range(m):
        r = len(p); i = next((i for i in range(r, n) if a[i][c]), None)
        if i is None: continue
        a[r], a[i] = a[i], a[r]; a[r] = [x / a[r][c] for x in a[r]]
        for i in range(n):
            if i != r: a[i] = [x - a[i][c] * y for x, y in zip(a[i], a[r])]
        p.append(c)
    return a, p

def ker(a):
    r, p = rref(a); m = len(a[0]); b = []
    for f in range(m):
        if f in p: continue
        v = [0] * m; v[f] = 1
        for i, c in enumerate(p): v[c] = -r[i][f]
        b.append(v)
    return b

def det(a):
    a = [[F(x) for x in r] for r in a]; n = len(a); s = F(1)
    for c in range(n):
        i = next((i for i in range(c, n) if a[i][c]), None)
        if i is None: return 0
        if i != c: a[c], a[i] = a[i], a[c]; s = -s
        s *= a[c][c]
        for i in range(c + 1, n): a[i] = [x - a[i][c] / a[c][c] * y for x, y in zip(a[i], a[c])]
    return s

inv = lambda a: [r[len(a):] for r in rref([list(r) + [int(i == j) for j in range(len(a))] for i, r in enumerate(a)])[0]]

In [7]:
# 3.1
V = [[1,2,1],[2,1,-1],[1,0,-1]]; r, p = rref(V)
print('3.1', Q(r), p, Q(ker(V)))                                                           

3.1 [[1, 0, -1], [0, 1, 1], [0, 0, 0]] [0, 1] [[1, -1, 1]]


In [8]:
# 3.2
M, U = np.array([[1,1,1,1],[1,0,0,-1]]), np.array([[1,0,-2,1],[0,1,-1,0]])
print('3.2', (M @ U.T).tolist(), len(ker(M.tolist())))   

3.2 [[0, 0], [0, 0]] 2


In [9]:
# 4.2
A = [[1,1,0],[0,1,1],[1,0,-1]]
print('4.2', Q(ker(A)), rref(A)[1])  

4.2 [[1, -1, 1]] [0, 1]


In [11]:
# 6.1
A = np.array([[1,0,2],[-1,3,1]])
print('6.1', (A @ [2,1,-1]).tolist(), (A @ A.T).tolist(), (A.T @ A).tolist())

6.1 [0, 0] [[5, 1], [1, 11]] [[2, -3, 1], [-3, 9, 3], [1, 3, 5]]


In [10]:
# 6.7
r, p = rref([[1,2,1,0],[2,4,0,2],[1,2,3,-2]])
print('6.7', Q(r), p)

6.7 [[1, 2, 0, 1], [0, 0, 1, -1], [0, 0, 0, 0]] [0, 2]


In [12]:
# 6.8
M = [[1,3,2,2],[1,4,1,0],[3,5,10,14],[2,5,5,6]]
print('6.8', Q(rref(M)[0]), Q(ker(M)), (np.array(M) @ [0,2,-8,5]).tolist())

6.8 [[1, 0, 5, 8], [0, 1, -1, -2], [0, 0, 0, 0], [0, 0, 0, 0]] [[-5, 1, 1, 0], [-8, 2, 0, 1]] [0, 0, 0, 0]


In [13]:
# 6.10
B = inv([[1,2,1],[0,1,1],[1,1,1]])
print('6.10', Q(B), I(np.array(B, float) @ [1,2,3]))

6.10 [[0, -1, 1], [1, 0, -1], [-1, 1, 1]] [1, -2, 4]


In [14]:
A = [[1,-1,1],[1,0,0],[1,1,1]]; B = np.array(inv(A), float)
print('6.15', int(det(A)), I(B @ [0,1,0]), I(B @ [1,2,5]))      # p = 1 - x^2, q = 2 + 2x + x^2


6.15 2 [1, 0, -1] [2, 2, 1]


In [15]:
print('6.22', int(det([[2,0,1,3],[1,1,0,2],[0,3,1,1],[1,0,2,1]])))

6.22 -1


In [17]:
A, B = R.normal(size=(2,3,3)); A *= np.cbrt(2 / d(A)); B *= np.cbrt(-3 / d(B))
C = A[[2,1,0]]; C[1] += 5 * C[0]
print('6.23', D([d(A@B), d(A.T@B@B), d(iv(A)@B), d(3*A), d(B@A@iv(B)), d(C)]))

6.23 [-6.0, 18.0, -1.5, 54.0, 2.0, -2.0]


In [18]:
def lu(a):
    u = np.array(a, float); n = len(u); l = np.eye(n)
    for k in range(n):
        for i in range(k + 1, n): l[i,k] = u[i,k] / u[k,k]; u[i] -= l[i,k] * u[k]
    return l, u

def solve(l, u, b):
    n = len(b); y = np.zeros(n); x = np.zeros(n)
    for i in range(n): y[i] = b[i] - l[i,:i] @ y[:i]
    for i in reversed(range(n)): x[i] = (y[i] - u[i,i+1:] @ x[i+1:]) / u[i,i]
    return y, x

In [19]:
l, u = lu([[1,2,1],[3,8,1],[0,4,1]]); y, x = solve(l, u, [2,10,3])
print('6.31', I(l), I(u), I(u.diagonal().prod()), I(y), I(x))

6.31 [[1, 0, 0], [3, 1, 0], [0, 2, 1]] [[1, 2, 1], [0, 2, -2], [0, 0, 5]] 10 [2, 4, -5] [1, 1, -1]


In [20]:
M = np.array([[2,1,1],[1,2,1],[1,1,2]], float); a, b, c, E = M[:1,:1], M[:1,1:], M[1:,:1], M[1:,1:]
ia = iv(a); S = E - c @ ia @ b; iS = iv(S)
Mi = np.block([[ia + ia @ b @ iS @ c @ ia, -ia @ b @ iS], [-iS @ c @ ia, iS]])
print('6.33', D(S), I(d(M)), I(4 * Mi), I(M @ Mi))

6.33 [[1.5, 0.5], [0.5, 1.5]] 4 [[3, -1, -1], [-1, 3, -1], [-1, -1, 3]] [[1, 0, 0], [0, 1, 0], [0, 0, 1]]


In [21]:
B = [[1,2,0,3],[2,4,1,8],[1,2,1,5]]; r, p = rref(B)
print('7.1', Q(r), p, Q(ker(B)))

7.1 [[1, 2, 0, 3], [0, 0, 1, 2], [0, 0, 0, 0]] [0, 2] [[-2, 1, 0, 0], [-3, 0, -2, 1]]


In [23]:
print('7.2', L.matrix_rank(R.normal(size=(4,6))), L.matrix_rank(R.normal(size=(5,3))))

7.2 4 3


In [24]:
Pm = [[2,1],[1,1]]
print('8.2', Q(inv(Pm)), (np.array([3,1]) @ Pm).tolist())         

8.2 [[1, -1], [-1, 2]] [7, 4]


In [25]:
A, s = np.array([[1,1,0],[0,1,-1]]), np.array([2,-1])
print('8.4', (s @ A).tolist(), s @ A @ [1,2,3], s @ (A @ [1,2,3]), np.array([e @ A for e in np.eye(2, dtype=int)]).T.tolist())

8.4 [2, 1, 1] 7 7 [[1, 0], [1, 1], [0, -1]]


In [26]:
f = lambda x, y: x - 2 * y
print('9.1', [f(3,1), f(1,0), f(5,3), f(1,1), f(1,2)])
print('9.4', Q(ker([[1,1,1,1],[1,-1,1,-1]])))

9.1 [1, 1, -1, -1, -3]
9.4 [[-1, 0, 1, 0], [0, -1, 0, 1]]


In [27]:
def sm(z, naive=False):
    e = np.exp(z if naive else z - z.max()); return e / e.sum()

In [28]:
z = np.array([1., 2., 3.])
with np.errstate(all='ignore'): print('9.6', D(sm(z + 1000, True)), D(sm(z + 1000)), D(sm(z)))

9.6 [nan, nan, nan] [0.09, 0.2447, 0.6652] [0.09, 0.2447, 0.6652]


In [29]:
A, P = np.array([[1,2],[3,2]]), np.array([[2,1],[3,-1]])
Ak = lambda k: np.array([[2*4**k + 3*(-1)**k, 2*4**k - 2*(-1)**k], [3*4**k - 3*(-1)**k, 3*4**k + 2*(-1)**k]]) / 5
print('10.1', D(np.sort(L.eigvals(A).real)), I(P @ np.diag([4,-1]) @ iv(P)), I(Ak(5)), L.matrix_power(A, 5).tolist())

10.1 [-1.0, 4.0] [[1, 2], [3, 2]] [[409, 410], [615, 614]] [[409, 410], [615, 614]]


In [30]:
N, E = np.array([[2,1,0],[0,2,0],[0,0,3]]), np.eye(3, dtype=int)
print('10.3', 3 - L.matrix_rank(N - 2*E), 3 - L.matrix_rank(N - 3*E), ((N - 2*E) @ (N - 3*E)).tolist())

10.3 1 1 [[0, -1, 0], [0, 0, 0], [0, 0, 0]]


In [31]:
A, x = np.array([[.7,.4],[.3,.6]]), np.array([1., 0.])
xk = lambda k: np.array([4,3]) / 7 + 3/7 * .3**k * np.array([1,-1])
print('10.9', D(A @ A @ x), D(xk(2)), D(L.matrix_power(A, 50) @ x))

10.9 [0.61, 0.39] [0.61, 0.39] [0.5714, 0.4286]


In [32]:
B, E = np.array([[1,1],[-2,4]]), np.eye(2, dtype=int); a, b = 1, 0
print('10.13', (B @ B - 5*B + 6*E).tolist(), D(iv(B)), D(-B/6 + 5*E/6), L.matrix_power(B, 3).tolist(), (19*B - 30*E).tolist())
for k in range(1, 6): print(' ', k, L.matrix_power(B, k).tolist(), (a*B + b*E).tolist()); a, b = 5*a + b, -6*a

10.13 [[0, 0], [0, 0]] [[0.6667, -0.1667], [0.3333, 0.1667]] [[0.6667, -0.1667], [0.3333, 0.1667]] [[-11, 19], [-38, 46]] [[-11, 19], [-38, 46]]
  1 [[1, 1], [-2, 4]] [[1, 1], [-2, 4]]
  2 [[-1, 5], [-10, 14]] [[-1, 5], [-10, 14]]
  3 [[-11, 19], [-38, 46]] [[-11, 19], [-38, 46]]
  4 [[-49, 65], [-130, 146]] [[-49, 65], [-130, 146]]
  5 [[-179, 211], [-422, 454]] [[-179, 211], [-422, 454]]


In [42]:
c, x, ft = np.array([1,1,0,0]), np.array([1,0,2,0]), np.fft.fft
C = np.array([[c[(j - k) % 4] for k in range(4)] for j in range(4)]); y = C @ x
print('10.15', C.tolist(), y.tolist(), D(ft(c)), I(ft(x).real), D(ft(y)), D(ft(c) * ft(x)), (C @ [1,-1,1,-1]).tolist())

10.15 [[1, 0, 0, 1], [1, 1, 0, 0], [0, 1, 1, 0], [0, 0, 1, 1]] [1, 1, 2, 2] [(2+0j), (1-1j), 0j, (1+1j)] [3, -1, 3, -1] [(6+0j), (-1+1j), 0j, (-1-1j)] [(6+0j), (-1+1j), 0j, (-1-1j)] [0, 0, 0, 0]


In [43]:
print('11.1', np.outer([1,2], [3,-1]).tolist(), L.matrix_rank(np.ones((2,2))), L.matrix_rank([[0,1],[-1,0]]))
T = np.arange(1, 25).reshape(2,3,4)

11.1 [[3, -1], [6, -2]] 1 2


In [44]:
print('11.2', 300*3*64*64, T[1,0,2], T[1,1,0])                   
x, y = R.normal(size=(2,2))

11.2 3686400 15 17


In [45]:
print('11.3', D([2*x[0]*y[0] - x[0]*y[1] - x[1]*y[0] + 4*x[1]*y[1], x @ np.array([[2,-1],[-1,4]]) @ y,
                 (x[0] - x[1]) * (y[0] + 2*y[1]), x @ np.array([[1,2],[-1,-2]]) @ y]))

11.3 [6.3535, 6.3535, 6.0437, 6.0437]


In [46]:
u, v, w = np.array([1,-1]), np.array([2,0,1]), np.array([1,3]); T = np.einsum('i,j,k->ijk', u, v, w)
print('11.4', np.outer(u, v).tolist(), T[...,0].tolist(), T[...,1].tolist(), (np.outer([1,0], [1,2]) + np.outer([0,1], [3,4])).tolist())

11.4 [[2, 0, 1], [-2, 0, -1]] [[2, 0, 1], [-2, 0, -1]] [[6, 0, 3], [-6, 0, -3]] [[1, 2], [3, 4]]


In [47]:
A, x = R.normal(size=(4,3)), R.normal(size=4); G, H = R.normal(size=(2,3,3)); X, W = R.normal(size=(5,6)), R.normal(size=(2,6))
print('11.5', eq(np.einsum('ij,i->j', A, x), A.T @ x),          # (n,)
      eq(np.einsum('ij,ij->', G, H), (G * H).sum()),             # scalar
      eq(np.einsum('ij,ji->i', G, H), np.diag(G @ H)),           # (n,), n^2 mults
      eq(np.einsum('ij,kj->ik', X, W), X @ W.T))                 # (N, m)

11.5 True True True True
